# Showing Unicode Characters in a Table

In the lecture, we have defined the *Unicode alphabet* as the set
$$ \Sigma_{\textrm{Unicode}} = \{ 0, 1, \cdots, 1\,114\,111 \}, $$
i.e. every character is identified with a natural number, its *code point*.  Code points are usually
written in hexadecimal notation using the prefix `U+`, so the largest code point is `U+10FFFF`.

This notebook implements a small program that displays a range of Unicode characters in a table.  For
every character, the table shows the *glyph*, i.e. the picture of the character, its code point, and
optionally its official Unicode name.

## Imports

- The module `unicodedata` gives access to the *Unicode Character Database*.  We use it to look up the
  name and the *general category* of a character.
- The module `html` provides the function `html.escape`, which we need since characters like `<` and `&`
  have a special meaning in HTML.
- The function `display` and the class `HTML` from the module `IPython.display` enable us to render an
  HTML string as a table inside the notebook.

In [1]:
import unicodedata
import html
from IPython.display import display, HTML

## Printable Characters

Not every code point can be displayed:
- Many code points are not yet assigned to a character.
- Some code points are *control codes*, e.g. the code point $10$ encodes a line feed.
- Other code points are reserved for special purposes, e.g. the *surrogates* in the range from `U+D800`
  to `U+DFFF`, which are used by the encoding UTF-16.

Every character belongs to a *general category*, which is a string consisting of two letters.  The first
letter gives the major class of the character, e.g. `L` for letters, `N` for numbers, `P` for
punctuation, and `C` for *other* characters.  All code points that cannot be displayed belong to one of the
categories `Cc` (control), `Cf` (format), `Cs` (surrogate), `Co` (private use), and `Cn` (unassigned).
Besides these, the space characters of category `Zs` are displayed as nothing.

The function `is_printable(c)` checks whether the code point `c` stands for a character that has a visible
glyph.  The function `chr(c)` converts the code point `c` into a string of length $1$, while the function
`unicodedata.category` returns the general category of this character.

In [2]:
def is_printable(c):
    category = unicodedata.category(chr(c))
    return category[0] != 'C' and category != 'Zs'

Let's test this function with the code points of the letter `A`, the space character, and the line feed.

In [3]:
is_printable(65), is_printable(32), is_printable(10)

(True, False, False)

## Formatting a Single Cell

The function `cell(c, names)` returns the HTML code of the two table cells that describe the code point `c`.
- The first cell contains the glyph, which is escaped with `html.escape`.  If the character is not
  printable, the cell is left empty and gets a grey background instead.  We do not use a placeholder symbol
  since every symbol we could choose is itself a Unicode character and might therefore occur in the table.
- The second cell contains the code point in the format `U+XXXX`.  The format specification `04X` writes
  the number `c` in hexadecimal notation with capital letters, using at least $4$ digits.
- If the flag `names` is `True`, the second cell shows the official name of the character below the code point, using a
  smaller font.  This name is returned by `unicodedata.name`.  The second argument of `unicodedata.name` is returned when a
  character has no name, which is the case, e.g., for control codes.  The tag `<br>` starts a new line.

In [4]:
def cell(c, names):
    if is_printable(c):
        glyph, style = html.escape(chr(c)), ''
    else:
        glyph, style = '', 'background:lightgrey; '
    code = f'<code>U+{c:04X}</code>'
    if names:
        name  = unicodedata.name(chr(c), 'no name')
        code += f'<br><span style="font-size:70%">{html.escape(name)}</span>'
    return (f'<td style="{style}font-size:150%; text-align:center">{glyph}</td>'
            f'<td style="text-align:left">{code}</td>')

## Building the Table

The function `unicode_table(first, last, columns, names)` returns an HTML table that shows all characters with
code points from `first` up to and including `last`.  Every row of the table shows `columns` characters.
If `names` is `True`, the table also shows the names of the characters.  For many characters, e.g. the
letters of the Latin alphabet, the names do not tell us anything new, so by default they are not shown.
- `codes` is the list of all code points that are to be shown.
- The header of the table contains the column titles `Glyph` and `Code` once for every character in a row.
- The list comprehension that computes `rows` splits the list `codes` into slices of length `columns`.
  Every slice is turned into one row of the table.  The last row might be shorter than the other rows.

In [5]:
def unicode_table(first, last, columns=8, names=False):
    codes  = list(range(first, last + 1))
    header = '<tr>' + '<th>Glyph</th><th>Code</th>' * columns + '</tr>'
    rows   = [ '<tr>' + ''.join(cell(c, names) for c in codes[i:i+columns]) + '</tr>'
               for i in range(0, len(codes), columns)
             ]
    return HTML('<table>' + header + ''.join(rows) + '</table>')

## Trying it Out

We start with the printable characters of the *ASCII alphabet*, which are the characters with the code
points from $32$ up to $126$.  Compare this table with Table 1.1 of the lecture notes.

In [6]:
display(unicode_table(32, 126))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
,U+0020,!,U+0021,"""",U+0022,#,U+0023,$,U+0024,%,U+0025,&,U+0026,',U+0027
(,U+0028,),U+0029,*,U+002A,+,U+002B,",",U+002C,-,U+002D,.,U+002E,/,U+002F
0,U+0030,1,U+0031,2,U+0032,3,U+0033,4,U+0034,5,U+0035,6,U+0036,7,U+0037
8,U+0038,9,U+0039,:,U+003A,;,U+003B,<,U+003C,=,U+003D,>,U+003E,?,U+003F
@,U+0040,A,U+0041,B,U+0042,C,U+0043,D,U+0044,E,U+0045,F,U+0046,G,U+0047
H,U+0048,I,U+0049,J,U+004A,K,U+004B,L,U+004C,M,U+004D,N,U+004E,O,U+004F
P,U+0050,Q,U+0051,R,U+0052,S,U+0053,T,U+0054,U,U+0055,V,U+0056,W,U+0057
X,U+0058,Y,U+0059,Z,U+005A,[,U+005B,\,U+005C,],U+005D,^,U+005E,_,U+005F
`,U+0060,a,U+0061,b,U+0062,c,U+0063,d,U+0064,e,U+0065,f,U+0066,g,U+0067
h,U+0068,i,U+0069,j,U+006A,k,U+006B,l,U+006C,m,U+006D,n,U+006E,o,U+006F


The Greek letters start at the code point `U+0391`.  Here, we also show the names of the characters.  Note that there is no character with the code point
`U+03A2`: this code point is not assigned, so its cell is grey.  The lower case letter `ς` at `U+03C2` is the variant of `σ` that
is used at the end of a word, so there is no need for a corresponding capital letter.

In [7]:
display(unicode_table(0x0391, 0x03C9, names=True))

Unicode also contains many mathematical symbols.  Some of the most important ones are found in the block
*Mathematical Operators*, which starts at the code point `U+2200`.

In [8]:
display(unicode_table(0x2200, 0x225F, names=True))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
∀,U+2200FOR ALL,∁,U+2201COMPLEMENT,∂,U+2202PARTIAL DIFFERENTIAL,∃,U+2203THERE EXISTS,∄,U+2204THERE DOES NOT EXIST,∅,U+2205EMPTY SET,∆,U+2206INCREMENT,∇,U+2207NABLA
∈,U+2208ELEMENT OF,∉,U+2209NOT AN ELEMENT OF,∊,U+220ASMALL ELEMENT OF,∋,U+220BCONTAINS AS MEMBER,∌,U+220CDOES NOT CONTAIN AS MEMBER,∍,U+220DSMALL CONTAINS AS MEMBER,∎,U+220EEND OF PROOF,∏,U+220FN-ARY PRODUCT
∐,U+2210N-ARY COPRODUCT,∑,U+2211N-ARY SUMMATION,−,U+2212MINUS SIGN,∓,U+2213MINUS-OR-PLUS SIGN,∔,U+2214DOT PLUS,∕,U+2215DIVISION SLASH,∖,U+2216SET MINUS,∗,U+2217ASTERISK OPERATOR
∘,U+2218RING OPERATOR,∙,U+2219BULLET OPERATOR,√,U+221ASQUARE ROOT,∛,U+221BCUBE ROOT,∜,U+221CFOURTH ROOT,∝,U+221DPROPORTIONAL TO,∞,U+221EINFINITY,∟,U+221FRIGHT ANGLE
∠,U+2220ANGLE,∡,U+2221MEASURED ANGLE,∢,U+2222SPHERICAL ANGLE,∣,U+2223DIVIDES,∤,U+2224DOES NOT DIVIDE,∥,U+2225PARALLEL TO,∦,U+2226NOT PARALLEL TO,∧,U+2227LOGICAL AND
∨,U+2228LOGICAL OR,∩,U+2229INTERSECTION,∪,U+222AUNION,∫,U+222BINTEGRAL,∬,U+222CDOUBLE INTEGRAL,∭,U+222DTRIPLE INTEGRAL,∮,U+222ECONTOUR INTEGRAL,∯,U+222FSURFACE INTEGRAL
∰,U+2230VOLUME INTEGRAL,∱,U+2231CLOCKWISE INTEGRAL,∲,U+2232CLOCKWISE CONTOUR INTEGRAL,∳,U+2233ANTICLOCKWISE CONTOUR INTEGRAL,∴,U+2234THEREFORE,∵,U+2235BECAUSE,∶,U+2236RATIO,∷,U+2237PROPORTION
∸,U+2238DOT MINUS,∹,U+2239EXCESS,∺,U+223AGEOMETRIC PROPORTION,∻,U+223BHOMOTHETIC,∼,U+223CTILDE OPERATOR,∽,U+223DREVERSED TILDE,∾,U+223EINVERTED LAZY S,∿,U+223FSINE WAVE
≀,U+2240WREATH PRODUCT,≁,U+2241NOT TILDE,≂,U+2242MINUS TILDE,≃,U+2243ASYMPTOTICALLY EQUAL TO,≄,U+2244NOT ASYMPTOTICALLY EQUAL TO,≅,U+2245APPROXIMATELY EQUAL TO,≆,U+2246APPROXIMATELY BUT NOT ACTUALLY EQUAL TO,≇,U+2247NEITHER APPROXIMATELY NOR ACTUALLY EQUAL TO
≈,U+2248ALMOST EQUAL TO,≉,U+2249NOT ALMOST EQUAL TO,≊,U+224AALMOST EQUAL OR EQUAL TO,≋,U+224BTRIPLE TILDE,≌,U+224CALL EQUAL TO,≍,U+224DEQUIVALENT TO,≎,U+224EGEOMETRICALLY EQUIVALENT TO,≏,U+224FDIFFERENCE BETWEEN


The *CJK Unified Ideographs* are Chinese characters, which are also used in Japanese and Korean.  This block
starts at `U+4E00`.  The first of these characters is `一`, the Chinese character for the number $1$.  We do
not show the names here, since they only contain the code point, e.g. `CJK UNIFIED IDEOGRAPH-4E00`.

In [9]:
display(unicode_table(0x4E00, 0x4E3F))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
一,U+4E00,丁,U+4E01,丂,U+4E02,七,U+4E03,丄,U+4E04,丅,U+4E05,丆,U+4E06,万,U+4E07
丈,U+4E08,三,U+4E09,上,U+4E0A,下,U+4E0B,丌,U+4E0C,不,U+4E0D,与,U+4E0E,丏,U+4E0F
丐,U+4E10,丑,U+4E11,丒,U+4E12,专,U+4E13,且,U+4E14,丕,U+4E15,世,U+4E16,丗,U+4E17
丘,U+4E18,丙,U+4E19,业,U+4E1A,丛,U+4E1B,东,U+4E1C,丝,U+4E1D,丞,U+4E1E,丟,U+4E1F
丠,U+4E20,両,U+4E21,丢,U+4E22,丣,U+4E23,两,U+4E24,严,U+4E25,並,U+4E26,丧,U+4E27
丨,U+4E28,丩,U+4E29,个,U+4E2A,丫,U+4E2B,丬,U+4E2C,中,U+4E2D,丮,U+4E2E,丯,U+4E2F
丰,U+4E30,丱,U+4E31,串,U+4E32,丳,U+4E33,临,U+4E34,丵,U+4E35,丶,U+4E36,丷,U+4E37
丸,U+4E38,丹,U+4E39,为,U+4E3A,主,U+4E3B,丼,U+4E3C,丽,U+4E3D,举,U+4E3E,丿,U+4E3F


Unicode also contains a large number of *emojis*.  The block *Emoticons* starts at `U+1F600`.  Note
that these code points are bigger than $2^{16} = 65\,536$, so they need more than $4$ hexadecimal digits.

In [10]:
display(unicode_table(0x1F600, 0x1F64F, names=True))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
😀,U+1F600GRINNING FACE,😁,U+1F601GRINNING FACE WITH SMILING EYES,😂,U+1F602FACE WITH TEARS OF JOY,😃,U+1F603SMILING FACE WITH OPEN MOUTH,😄,U+1F604SMILING FACE WITH OPEN MOUTH AND SMILING EYES,😅,U+1F605SMILING FACE WITH OPEN MOUTH AND COLD SWEAT,😆,U+1F606SMILING FACE WITH OPEN MOUTH AND TIGHTLY-CLOSED EYES,😇,U+1F607SMILING FACE WITH HALO
😈,U+1F608SMILING FACE WITH HORNS,😉,U+1F609WINKING FACE,😊,U+1F60ASMILING FACE WITH SMILING EYES,😋,U+1F60BFACE SAVOURING DELICIOUS FOOD,😌,U+1F60CRELIEVED FACE,😍,U+1F60DSMILING FACE WITH HEART-SHAPED EYES,😎,U+1F60ESMILING FACE WITH SUNGLASSES,😏,U+1F60FSMIRKING FACE
😐,U+1F610NEUTRAL FACE,😑,U+1F611EXPRESSIONLESS FACE,😒,U+1F612UNAMUSED FACE,😓,U+1F613FACE WITH COLD SWEAT,😔,U+1F614PENSIVE FACE,😕,U+1F615CONFUSED FACE,😖,U+1F616CONFOUNDED FACE,😗,U+1F617KISSING FACE
😘,U+1F618FACE THROWING A KISS,😙,U+1F619KISSING FACE WITH SMILING EYES,😚,U+1F61AKISSING FACE WITH CLOSED EYES,😛,U+1F61BFACE WITH STUCK-OUT TONGUE,😜,U+1F61CFACE WITH STUCK-OUT TONGUE AND WINKING EYE,😝,U+1F61DFACE WITH STUCK-OUT TONGUE AND TIGHTLY-CLOSED EYES,😞,U+1F61EDISAPPOINTED FACE,😟,U+1F61FWORRIED FACE
😠,U+1F620ANGRY FACE,😡,U+1F621POUTING FACE,😢,U+1F622CRYING FACE,😣,U+1F623PERSEVERING FACE,😤,U+1F624FACE WITH LOOK OF TRIUMPH,😥,U+1F625DISAPPOINTED BUT RELIEVED FACE,😦,U+1F626FROWNING FACE WITH OPEN MOUTH,😧,U+1F627ANGUISHED FACE
😨,U+1F628FEARFUL FACE,😩,U+1F629WEARY FACE,😪,U+1F62ASLEEPY FACE,😫,U+1F62BTIRED FACE,😬,U+1F62CGRIMACING FACE,😭,U+1F62DLOUDLY CRYING FACE,😮,U+1F62EFACE WITH OPEN MOUTH,😯,U+1F62FHUSHED FACE
😰,U+1F630FACE WITH OPEN MOUTH AND COLD SWEAT,😱,U+1F631FACE SCREAMING IN FEAR,😲,U+1F632ASTONISHED FACE,😳,U+1F633FLUSHED FACE,😴,U+1F634SLEEPING FACE,😵,U+1F635DIZZY FACE,😶,U+1F636FACE WITHOUT MOUTH,😷,U+1F637FACE WITH MEDICAL MASK
😸,U+1F638GRINNING CAT FACE WITH SMILING EYES,😹,U+1F639CAT FACE WITH TEARS OF JOY,😺,U+1F63ASMILING CAT FACE WITH OPEN MOUTH,😻,U+1F63BSMILING CAT FACE WITH HEART-SHAPED EYES,😼,U+1F63CCAT FACE WITH WRY SMILE,😽,U+1F63DKISSING CAT FACE WITH CLOSED EYES,😾,U+1F63EPOUTING CAT FACE,😿,U+1F63FCRYING CAT FACE
🙀,U+1F640WEARY CAT FACE,🙁,U+1F641SLIGHTLY FROWNING FACE,🙂,U+1F642SLIGHTLY SMILING FACE,🙃,U+1F643UPSIDE-DOWN FACE,🙄,U+1F644FACE WITH ROLLING EYES,🙅,U+1F645FACE WITH NO GOOD GESTURE,🙆,U+1F646FACE WITH OK GESTURE,🙇,U+1F647PERSON BOWING DEEPLY
🙈,U+1F648SEE-NO-EVIL MONKEY,🙉,U+1F649HEAR-NO-EVIL MONKEY,🙊,U+1F64ASPEAK-NO-EVIL MONKEY,🙋,U+1F64BHAPPY PERSON RAISING ONE HAND,🙌,U+1F64CPERSON RAISING BOTH HANDS IN CELEBRATION,🙍,U+1F64DPERSON FROWNING,🙎,U+1F64EPERSON WITH POUTING FACE,🙏,U+1F64FPERSON WITH FOLDED HANDS


## Cuneiform

Finally, Unicode also covers historical scripts.  *Cuneiform* is one of the oldest known writing systems.  It
was used in Mesopotamia for more than 3000 years, starting around 3400 BC.  The characters were pressed into
clay tablets with a reed stylus, which gives them their wedge-shaped look (Latin *cuneus* means *wedge*).  The
block *Cuneiform* starts at `U+12000`.  The next cell shows the first $100$ characters of this block.  We do
not show their names: names like `CUNEIFORM SIGN A` are only the labels that Assyriologists use for these
signs, not translations.  Most signs have several readings, which depend on the language and the context.

**Note:** Your browser needs a font that contains these characters, e.g. *Noto Sans Cuneiform*.  Otherwise,
the characters are shown as empty boxes.

In [11]:
display(unicode_table(0x12000, 0x12063))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
𒀀,U+12000,𒀁,U+12001,𒀂,U+12002,𒀃,U+12003,𒀄,U+12004,𒀅,U+12005,𒀆,U+12006,𒀇,U+12007
𒀈,U+12008,𒀉,U+12009,𒀊,U+1200A,𒀋,U+1200B,𒀌,U+1200C,𒀍,U+1200D,𒀎,U+1200E,𒀏,U+1200F
𒀐,U+12010,𒀑,U+12011,𒀒,U+12012,𒀓,U+12013,𒀔,U+12014,𒀕,U+12015,𒀖,U+12016,𒀗,U+12017
𒀘,U+12018,𒀙,U+12019,𒀚,U+1201A,𒀛,U+1201B,𒀜,U+1201C,𒀝,U+1201D,𒀞,U+1201E,𒀟,U+1201F
𒀠,U+12020,𒀡,U+12021,𒀢,U+12022,𒀣,U+12023,𒀤,U+12024,𒀥,U+12025,𒀦,U+12026,𒀧,U+12027
𒀨,U+12028,𒀩,U+12029,𒀪,U+1202A,𒀫,U+1202B,𒀬,U+1202C,𒀭,U+1202D,𒀮,U+1202E,𒀯,U+1202F
𒀰,U+12030,𒀱,U+12031,𒀲,U+12032,𒀳,U+12033,𒀴,U+12034,𒀵,U+12035,𒀶,U+12036,𒀷,U+12037
𒀸,U+12038,𒀹,U+12039,𒀺,U+1203A,𒀻,U+1203B,𒀼,U+1203C,𒀽,U+1203D,𒀾,U+1203E,𒀿,U+1203F
𒁀,U+12040,𒁁,U+12041,𒁂,U+12042,𒁃,U+12043,𒁄,U+12044,𒁅,U+12045,𒁆,U+12046,𒁇,U+12047
𒁈,U+12048,𒁉,U+12049,𒁊,U+1204A,𒁋,U+1204B,𒁌,U+1204C,𒁍,U+1204D,𒁎,U+1204E,𒁏,U+1204F
